In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from typing import List
from pyspark.sql import DataFrame
from pyspark.sql import Window

In [0]:
import os
import sys

In [0]:
current_dir = os.getcwd()

sys.path.append(current_dir)
current_dir

In [0]:
class transforamtion:
        
    def dedup(self,df:DataFrame,dedu_cols:List,cdc:str):
        
        
        df = df.withColumn("dedukey",concat(*dedu_cols))
        df = df.withColumn("deduCounts",row_number()\
            .over(Window.partitionBy("dedukey")\
                .orderBy(desc(cdc))\
                                        ))
        df = df.filter(col("deduCounts")==1)
        df = df.drop('dedukey',"deduCounts")

        return df
    
    def process_timestamp(self,df):

        df = df.withColumn("process_timestamp",current_timestamp())

        return df
    
    def upsart(self,df,key_cols,table,cdc):

        merge_condistion = " AND ".join([f"sr.{i} = trg.{i}" for i in key_cols ])
        dlt_obj = DeltaTable.forName(spark,f"pysparkdbt.silver.{table}")
        dlt_obj.alias("trg").merge(df.alias("src"),merge_condistion)\
            .whenMatchedUpdateAll(condition= f"src.{cdc} >= trg.{cdc}")\
            .whenNotMatchedInsertAll()\
            .execute()

        return 1    





#### ***CUSTOMERS***

In [0]:
df_cust = spark.read.table("pysparkdbt.bronze.customers")

In [0]:
display(df_cust)

In [0]:
df_cust =  df_cust.withColumn("domain", split(col('email'),'@')[1])
display(df_cust)

In [0]:
df_cust =  df_cust.withColumn("phone_number", regexp_replace("phone_number",r"[^0-9]",""))
display(df_cust)

In [0]:
df_cust =  df_cust.withColumn("Full_Name", concat_ws(" ", col("first_name"), col("last_name")))
df_cust = df_cust.drop("first_name", "last_name")
display(df_cust)

In [0]:
cust_obj = transforamtion()

cust_df_trns = cust_obj.dedup(df_cust,['customer_id'],'last_updated_timestamp')
display(cust_df_trns)

In [0]:
df_cust = cust_obj.process_timestamp(cust_df_trns)
display(df_cust)

In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists("pysparkdbt.silver.customers"):
    
    df_cust.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.customers")

else:

    cust_obj.upsart(df_cust,['customer_id'],'customers','last_updated_timestamp')
      


In [0]:
%sql
SELECT * FROM pysparkdbt.silver.customers

### Drivers

In [0]:
df_drivers = spark.read.table("pysparkdbt.bronze.drivers")
display(df_drivers)

In [0]:
df_drivers =  df_drivers.withColumn("phone_number", regexp_replace("phone_number",r"[^0-9]",""))

In [0]:
df_drivers =  df_drivers.withColumn("Full_Name", concat_ws(" ", col("first_name"), col("last_name")))
df_drivers = df_drivers.drop("first_name", "last_name")


In [0]:
display(df_drivers)

In [0]:
driver_obj = transforamtion()

In [0]:
df_drivers = driver_obj.dedup(df_drivers,['driver_id'],'last_updated_timestamp')

In [0]:
df_drivers = driver_obj.process_timestamp(df_drivers)

In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.drivers"):
    
    df_drivers.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.drivers")

else:

    driver_obj.upsart(df_drivers,['driver_id'],'drivers','last_updated_timestamp')

In [0]:
%sql
SELECT * FROM pysparkdbt.silver.drivers

#### LOCATIONS

In [0]:
df_loc = spark.read.table("pysparkdbt.bronze.locations")
display(df_loc)

In [0]:
loc_obj = transforamtion()

In [0]:
df_loc = loc_obj.dedup(df_loc,['location_id'],'last_updated_timestamp')
df_loc = loc_obj.process_timestamp(df_loc)


In [0]:
if not spark.catalog.tableExists("pysparkdbt.silver.locations"):
    
    df_loc.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.locations")

else:   
    loc_obj.upsart(df_loc,['location_id'],'locations','last_updated_timestamp')

In [0]:
%sql
select * from pysparkdbt.silver.locations

### PAYMENTS

In [0]:
df_payments = spark.read.table("pysparkdbt.bronze.payments")
display(df_payments)

In [0]:
df_payments = df_payments.withColumn('online_payment_status', 
                                           when( ((col('payment_method')=='Card') & (col('payment_status')=='Success')), "online-success")
                                           .when( ((col('payment_method')=='Card') & (col('payment_status')=='Failed')), "online-failed")
                                           .when( ((col('payment_method')=='Card') & (col('payment_status')=='Pending')), "online-pending")
                                           .otherwise("offline")
                                           )
display(df_payments)                                           

In [0]:

payment_obj = transforamtion()
df_payments = payment_obj.dedup(df_payments,['payment_id'],'last_updated_timestamp')
df_payments = payment_obj.process_timestamp(df_payments)
if not spark.catalog.tableExists("pysparkdbt.silver.payments"):
    
    df_payments.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.payments")        
else:
    payment_obj.upsart(df_payments,['payment_id'],'payment','last_updated_timestamp')

In [0]:
%sql
select * from pysparkdbt.silver.payments

### VEHICLES

In [0]:
df_vehichle = spark.read.table("pysparkdbt.bronze.vehicles")
display(df_vehichle)

In [0]:
df_vehichle = df_vehichle.withColumn("make", upper(col("make")))
display(df_vehichle)

In [0]:
vehicle_obj = transforamtion()
df_vehichle = vehicle_obj.dedup(df_vehichle,['vehicle_id'],'last_updated_timestamp')
df_vehichle = vehicle_obj.process_timestamp(df_vehichle)        
if not spark.catalog.tableExists("pysparkdbt.silver.vehicles"):
    
    df_vehichle.write.format("delta")\
        .mode("append")\
        .saveAsTable("pysparkdbt.silver.vehicles")
else:
    vehicle_obj.upsart(df_vehichle,['vehicle_id'],'vehicles','last_updated_timestamp')

In [0]:
%sql
select * from pysparkdbt.silver.vehicles